# CAEclust — Deep Autoencoder-Based Document Clustering

This notebook implements an **unsupervised document-clustering pipeline** on the
[Reuters newswire dataset](https://keras.io/api/datasets/reuters/). The goal is to
group news articles into topics **without using their labels during training**, and
then measure how well the discovered clusters match the real topics.

### How it works

The pipeline has four stages:

1. **Text → vectors.** Each article is turned into a sequence of word vectors using
   pretrained [GloVe](https://nlp.stanford.edu/projects/glove/) embeddings.
2. **Representation learning.** An **LSTM autoencoder** compresses each article into a
   compact, fixed-length embedding by learning to reconstruct its own input.
3. **Affinity construction.** A **landmark-based** similarity (affinity) matrix is built
   from those embeddings. Using a small number of landmarks keeps this step scalable.
4. **Ensemble clustering.** Truncated SVD followed by K-Means produces the final
   cluster assignments, which are evaluated against the true labels with the
   **Normalized Mutual Information (NMI)** score.

> **Tip:** Run the cells top to bottom. A GPU is recommended (Runtime → Change runtime
> type → GPU in Google Colab) since training the LSTM autoencoder on CPU is slow.

## 1. Setup and dependencies

We import the libraries we need and enable GPU memory growth if a GPU is available.
Memory growth lets TensorFlow allocate GPU memory gradually instead of grabbing it all
at once.

In [ ]:
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt

from sklearn.cluster import KMeans
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.metrics import normalized_mutual_info_score
from sklearn.decomposition import TruncatedSVD
from sklearn.manifold import TSNE
from sklearn.preprocessing import normalize
from scipy.sparse import csr_matrix

from tensorflow.keras.datasets import reuters
from tensorflow.keras.preprocessing.sequence import pad_sequences

# Enable GPU memory growth if a GPU is present.
gpus = tf.config.list_physical_devices("GPU")
if gpus:
    try:
        tf.config.experimental.set_memory_growth(gpus[0], True)
        print("GPU enabled:", gpus[0].name)
    except RuntimeError as e:
        print("GPU configuration error:", e)
else:
    print("No GPU found - running on CPU (training will be slower).")

## 2. Download the GloVe embeddings

We use the **100-dimensional** GloVe vectors trained on 6 billion tokens
(`glove.6B.100d.txt`). The archive is ~822 MB, so this step may take a few minutes the
first time.

This cell must run **before** we load the embeddings further down — that ordering is
what makes the notebook reproducible from top to bottom.

In [ ]:
# Download and unzip GloVe (skips the download if the file already exists).
import os

if not os.path.exists("glove.6B.100d.txt"):
    !wget -q http://nlp.stanford.edu/data/glove.6B.zip
    !unzip -o glove.6B.zip
else:
    print("GloVe embeddings already present - skipping download.")

## 3. Load the GloVe embeddings

`load_glove_embeddings` reads the GloVe text file and builds an **embedding matrix**:
row `i` holds the vector for the word whose Reuters index is `i`. Words that have no
GloVe vector are left as zeros.

In [ ]:
def load_glove_embeddings(file_path, word_index, embedding_dim=100):
    """Build an embedding matrix aligned to the dataset's word index.

    Args:
        file_path:     Path to a GloVe ``.txt`` file (e.g. ``glove.6B.100d.txt``).
        word_index:    Mapping of ``word -> integer index`` from the dataset.
        embedding_dim: Dimensionality of the GloVe vectors.

    Returns:
        A ``(vocab_size, embedding_dim)`` NumPy array of word vectors.
    """
    # Read every word vector from the GloVe file into a dictionary.
    embeddings_index = {}
    with open(file_path, encoding="utf-8") as f:
        for line in f:
            values = line.split()
            word = values[0]
            coefs = np.asarray(values[1:], dtype="float32")
            embeddings_index[word] = coefs

    # Place each known word's vector at its index in the matrix.
    vocab_size = len(word_index) + 1
    embedding_matrix = np.zeros((vocab_size, embedding_dim))
    for word, i in word_index.items():
        embedding_vector = embeddings_index.get(word)
        if embedding_vector is not None:
            embedding_matrix[i] = embedding_vector

    print(f"Loaded embedding matrix with shape {embedding_matrix.shape}")
    return embedding_matrix

## 4. The CAEclust model

`CAEclust` bundles the whole pipeline into one class. The key methods are:

| Method | What it does |
| --- | --- |
| `build_autoencoder` | Defines a symmetric **LSTM encoder–decoder**. The encoder squeezes each article down to an `encoding_dim`-sized vector; the decoder tries to reconstruct the original sequence. |
| `train_autoencoder` | Trains the autoencoder (with early stopping) and returns the learned encodings. |
| `construct_affinity_matrix` | Picks `n_landmarks` representative points with K-Means, then builds a **sparse landmark-based affinity matrix** from cosine similarities. |
| `ensemble_clustering` | Reduces the affinity matrix with Truncated SVD and runs K-Means to get final cluster labels. |
| `fit` | Runs every stage end to end, reports the **NMI** score, and draws the diagnostic plots. |

The plotting helpers (`plot_training_loss`, `plot_tsne_clusters`) visualize the training
curve and a 2-D t-SNE projection of true vs. predicted clusters.

In [ ]:
class CAEclust:
    """Autoencoder + landmark-based ensemble clustering for documents."""

    def __init__(self, n_clusters=10, n_landmarks=300, encoding_dim=64):
        self.n_clusters = n_clusters
        self.n_landmarks = n_landmarks
        self.encoding_dim = encoding_dim
        self.autoencoder = None
        self.encoder = None
        self.history = None

    # ------------------------------------------------------------------
    # Model definition
    # ------------------------------------------------------------------
    def build_autoencoder(self, input_dim, max_seq_length=500, embedding_dim=100):
        """Build a symmetric LSTM encoder-decoder and return (autoencoder, encoder)."""
        print("Building the LSTM autoencoder...")
        input_layer = tf.keras.layers.Input(shape=(max_seq_length, embedding_dim))

        # Encoder: sequence -> compact fixed-length vector.
        x = tf.keras.layers.LSTM(128, return_sequences=True)(input_layer)
        x = tf.keras.layers.LSTM(64, return_sequences=False)(x)
        x = tf.keras.layers.Dense(128, activation="relu")(x)
        encoded = tf.keras.layers.Dense(self.encoding_dim, activation="tanh")(x)

        # Decoder: compact vector -> reconstructed sequence.
        x = tf.keras.layers.RepeatVector(max_seq_length)(encoded)
        x = tf.keras.layers.LSTM(64, return_sequences=True)(x)
        x = tf.keras.layers.LSTM(128, return_sequences=True)(x)
        decoded = tf.keras.layers.TimeDistributed(
            tf.keras.layers.Dense(embedding_dim, activation="linear")
        )(x)

        autoencoder = tf.keras.models.Model(input_layer, decoded)
        encoder = tf.keras.models.Model(input_layer, encoded)
        autoencoder.compile(optimizer="adam", loss="mse")

        print("Autoencoder built.")
        return autoencoder, encoder

    # ------------------------------------------------------------------
    # Training
    # ------------------------------------------------------------------
    def train_autoencoder(self, X_train, batch_size=64, epochs=100):
        """Train the autoencoder and return the learned encodings for X_train."""
        print("Training the autoencoder...")
        autoencoder, encoder = self.build_autoencoder(X_train.shape[-1])

        # Reconstruct the input, so inputs and targets are identical.
        dataset = (
            tf.data.Dataset.from_tensor_slices((X_train, X_train))
            .shuffle(10000)
            .batch(batch_size)
            .prefetch(tf.data.AUTOTUNE)
        )

        early_stop = tf.keras.callbacks.EarlyStopping(
            monitor="loss", patience=10, restore_best_weights=True
        )
        history = autoencoder.fit(dataset, epochs=epochs, callbacks=[early_stop], verbose=1)

        self.history = history
        self.autoencoder = autoencoder
        self.encoder = encoder
        print("Autoencoder training complete.")
        return encoder.predict(X_train)

    # ------------------------------------------------------------------
    # Affinity matrix
    # ------------------------------------------------------------------
    def construct_affinity_matrix(self, encodings):
        """Build a sparse landmark-based affinity matrix from the encodings."""
        print("Constructing the affinity matrix...")
        encodings = normalize(encodings)

        # Use a small set of landmarks to keep the affinity matrix scalable.
        kmeans = KMeans(n_clusters=self.n_landmarks, random_state=0).fit(encodings)
        landmarks = kmeans.cluster_centers_

        # Row-normalized cosine similarity of each point to every landmark.
        Z = cosine_similarity(encodings, landmarks)
        Z /= Z.sum(axis=1, keepdims=True) + 1e-10

        # Low-rank affinity: S = Z @ Z.T (kept sparse for efficiency).
        S = csr_matrix(Z) @ csr_matrix(Z.T)
        print(f"Affinity matrix shape: {S.shape}")
        return S

    # ------------------------------------------------------------------
    # Final clustering
    # ------------------------------------------------------------------
    def ensemble_clustering(self, S):
        """Reduce the affinity matrix with SVD, then cluster with K-Means."""
        print("Performing ensemble clustering...")
        svd = TruncatedSVD(n_components=self.n_clusters, random_state=0)
        U = svd.fit_transform(S)

        kmeans = KMeans(n_clusters=self.n_clusters, random_state=0).fit(U)
        print("Clustering complete.")
        return kmeans.labels_

    # ------------------------------------------------------------------
    # Visualization
    # ------------------------------------------------------------------
    def plot_training_loss(self):
        """Plot the autoencoder's training-loss curve."""
        if self.history:
            plt.figure(figsize=(8, 5))
            plt.plot(self.history.history["loss"], label="Training loss")
            plt.xlabel("Epochs")
            plt.ylabel("Loss")
            plt.title("Autoencoder training-loss curve")
            plt.legend()
            plt.grid(True)
            plt.show()

    def plot_tsne_clusters(self, encodings, labels_true, labels_pred):
        """Compare true labels vs. predicted clusters in 2-D using t-SNE."""
        tsne = TSNE(n_components=2, random_state=0)
        encodings_2d = tsne.fit_transform(encodings)

        plt.figure(figsize=(12, 5))

        plt.subplot(1, 2, 1)
        plt.scatter(encodings_2d[:, 0], encodings_2d[:, 1], c=labels_true, cmap="jet", s=10)
        plt.title("True labels (t-SNE)")
        plt.grid(True)

        plt.subplot(1, 2, 2)
        plt.scatter(encodings_2d[:, 0], encodings_2d[:, 1], c=labels_pred, cmap="jet", s=10)
        plt.title("Predicted clusters (t-SNE)")
        plt.grid(True)

        plt.show()

    # ------------------------------------------------------------------
    # Full pipeline
    # ------------------------------------------------------------------
    def fit(self, X, y_true):
        """Run the full pipeline and return the predicted cluster labels."""
        print("Starting the CAEclust pipeline...")
        encodings = self.train_autoencoder(X)
        S = self.construct_affinity_matrix(encodings)
        labels = self.ensemble_clustering(S)

        nmi = normalized_mutual_info_score(y_true, labels)
        print(f"NMI score: {nmi:.4f}")

        self.plot_training_loss()
        self.plot_tsne_clusters(encodings, y_true, labels)
        return labels

## 5. Load and preprocess the Reuters dataset

We load the Reuters articles (keeping the 5,000 most frequent words), build the GloVe
embedding matrix, pad every article to a fixed length of 500 tokens, and finally turn
each padded sequence into a `(500, 100)` array of word vectors. The result,
`X_train_embedded`, has shape `(num_articles, 500, 100)` — the input format the
autoencoder expects.

In [ ]:
# Load the Reuters dataset (we only need the training split here).
print("Loading the Reuters dataset...")
(X_train, y_train), (_, _) = reuters.load_data(num_words=5000)

# Build the embedding matrix from GloVe.
word_index = reuters.get_word_index()
embedding_matrix = load_glove_embeddings("glove.6B.100d.txt", word_index, embedding_dim=100)

# Pad all sequences to the same length.
MAX_SEQ_LENGTH = 500
EMBEDDING_DIM = 100
X_train = pad_sequences(X_train, maxlen=MAX_SEQ_LENGTH)

# Map each word index to its GloVe vector.
X_train_embedded = np.zeros((X_train.shape[0], MAX_SEQ_LENGTH, EMBEDDING_DIM))
for i, seq in enumerate(X_train):
    for j, word_idx in enumerate(seq):
        if word_idx < embedding_matrix.shape[0]:
            X_train_embedded[i, j] = embedding_matrix[word_idx]

print(f"Embedded dataset shape: {X_train_embedded.shape}")

## 6. Run the clustering pipeline

We instantiate `CAEclust` and call `fit`. This trains the autoencoder, builds the
affinity matrix, runs ensemble clustering, prints the NMI score, and displays the
training-loss and t-SNE plots.

`n_clusters=10` is chosen here for illustration; the full Reuters dataset has 46 topics,
so you may want to experiment with that value.

In [ ]:
caeclust = CAEclust(n_clusters=10, n_landmarks=300, encoding_dim=64)
labels = caeclust.fit(X_train_embedded, y_train)
print("Final clustering labels:", labels)